# Creación de Dataset de Películas 

El dataset utilizado contiene **32,000,204 calificaciones** y **2,000,072 etiquetas** asignadas por **200,948 usuarios** a **87,585 películas** del conjunto de datos **MovieLens**. Las interacciones fueron registradas entre el **9 de enero de 1995** y el **12 de octubre de 2023**.

El conjunto de datos se compone de los siguientes archivos:

- `ratings.parquet`: contiene las calificaciones otorgadas por los usuarios a las películas. Sus columnas son:
  - `userId`
  - `movieId`
  - `rating`
  - `timestamp`

- `links.csv`: relaciona los identificadores de MovieLens con otras bases de datos externas. Contiene las siguientes columnas:
  - `movieId`: identificador interno de MovieLens.
  - `imdbId`: identificador de la película en IMDb.
  - `tmdbId`: identificador de la película en TMDB.

- `movies.csv`: contiene información básica de las películas:
  - `movieId`
  - `title`
  - `genres`

- `tags.csv`: contiene etiquetas asignadas libremente por los usuarios:
  - `userId`
  - `movieId`
  - `tag`
  - `timestamp`

Sin embargo, se decidió no utilizar el archivo `movies.csv`, ya que la información que proporciona sobre las películas es limitada. En su lugar, se obtendrán datos más completos mediante la API de TMDB, los cuales se almacenarán en el archivo `movies_metadata.parquet`.

De igual manera, el archivo `tags.csv` no será utilizado, debido a que las etiquetas fueron generadas libremente por los usuarios y no pasaron por ningún proceso de validación o estandarización. Esto podría introducir ruido e inconsistencias en el sistema recomendador.

In [1]:
import pandas as pd
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import warnings
warnings.filterwarnings('ignore')

import os
from dotenv import load_dotenv
# Load the environment variables .env
load_dotenv()

True

Del archvio links.csv solo conservaremos `movieId` y `tmdbId` del archivo `tags.csv`, ya que `imdbId` no se utilizará en el análisis.

In [2]:
mdf = pd.read_csv('../data/raw/links.csv')
mdf.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [3]:
mdf.shape

(87585, 3)

In [4]:
# Check how many null values we have
mdf.isna().sum()

movieId      0
imdbId       0
tmdbId     124
dtype: int64

In [5]:
# Check if there are duplicated values 
mdf[mdf.duplicated(subset=['tmdbId'], keep=False)].sort_values(by='tmdbId').head(10)

,movieId,imdbId,tmdbId
5892,6003,290538,4912.0
34340,144606,270288,4912.0
10103,34330,368089,9775.0
49943,178755,376800,9775.0
4138,4241,266860,10991.0
47959,174533,235679,10991.0
5562,5672,313487,12600.0
47964,174543,287635,12600.0
61157,202599,1016268,13020.0
9897,33154,413845,13020.0


Realizaremos las siguientes acciones de limpieza antes de consultar la API de **TMDB** para extraer más datos de las películas:

- Eliminar entradas duplicadas por `tmdbId`
- Eliminar la columna `imdbId`, ya que no se utilizará
- Eliminar filas sin `tmdbId` válido
- Convertir `tmdbId` a entero para consistencia
- Renombrar la columna `tmdbId` a `id` para facilitar los merges

In [6]:
mdf.drop_duplicates(subset=['tmdbId'], inplace=True) 
mdf.drop(columns='imdbId', inplace=True)  
mdf.dropna(subset=['tmdbId'], inplace=True)  
mdf['tmdbId'] = mdf['tmdbId'].astype('int')
mdf.rename(columns={'tmdbId': 'id'}, inplace=True)
mdf.head()  

,movieId,id
0,1,862
1,2,8844
2,3,15602
3,4,31357
4,5,11862


## Descarga de Detalles con la API de TMDB

Para enriquecer este dataset con información útil para el sistema de recomendación —como título, géneros, sinopsis, director, reparto y puntuación— utilizaremos la **API de TMDB** para descargar los metadatos completos de cada película.

Los campos que nos interesan para el sistema de recomendación son:
- `title`, `genres`, `overview`, `tagline` — para el recomendador basado en contenido
- `vote_average`, `vote_count` — para calcular el score de popularidad
- `poster_path`, `backdrop_path` — para la interfaz de usuario
- `release_date`, `runtime` — para filtros y presentación

In [7]:
ids = mdf['id'].values
ids.shape

(87425,)

In [8]:
def fetch_movie(movie_id, api_key):
    """
    Function to fetch data for a single movie from TMDB API
    Args:
        movie_id (int): ID of the movie
        api_key (str): API key for authentication

    Returns:
        dict: Movie data if request is successful, None if failed
    """
    url = f"https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            return response.json()
        else:
            return None  # Return None if response code is not 200 (success)
    
    except requests.exceptions.RequestException as e:
        # Handle any network-related or request errors
        print(f"Error fetching data for movie ID {movie_id}: {e}")
        return None

In [9]:
def fetch_movies(ids, api_key):
    """
    Function to fetch movie data for multiple movie IDs using concurrent requests
    Args:
        ids (list): List of movie IDs to fetch
        api_key (str): API key for authentication
    
    Returns:
        tuple: A tuple containing two lists:
            - List of successfully fetched movies (as JSON)
            - List of movie IDs for which fetching data failed
    """
    id_errors = [] 
    movies = [] 

    # Using ThreadPoolExecutor to send multiple requests concurrently
    with ThreadPoolExecutor(max_workers=10) as executor:
        # Submitting the fetch_movie function to the executor for each movie ID
        futures = {executor.submit(fetch_movie, movie_id, api_key): movie_id for movie_id in ids}

        # Processing results as they complete
        for future in as_completed(futures):
            movie_id = futures[future]
            movie_data = future.result()
            
            if movie_data:
                movies.append(movie_data)
            else:
                id_errors.append(movie_id)
            
            # Adding a small sleep time to avoid hitting API rate limits
            time.sleep(0.005)

    return movies, id_errors

In [10]:
api_key = os.getenv('tmdb_api_key')
movies, id_errors = fetch_movies(ids, api_key)

print(f"Fetched {len(movies)} movies")
print(f"Failed to fetch {len(id_errors)} movies")

Fetched 86242 movies
Failed to fetch 1183 movies


## Conversión a DataFrame

Convertimos las películas descargadas exitosamente en un DataFrame. Posteriormente, lo fusionaremos con los IDs originales de MovieLens para mantener la consistencia del pipeline.

In [11]:
fetched = pd.DataFrame(data=movies)
fetched.head().transpose()

,0,1,2,3,4
adult,False,False,False,False,False
backdrop_path,/fIWsCpYR9iGDMSbMTSAzy8L7Kg5.jpg,/jP8lHNHD89xaRPfAdyz5KEVYcSb.jpg,/xKsnZDERG1dk95wuZ5q9iks3OL3.jpg,/oMGV48EGhsNavC1PL8HMeWs5Udq.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg
belongs_to_collection,"{'id': 645, 'name': 'James Bond Collection', '...",None,"{'id': 1048282, 'name': 'Heat Collection', 'po...",None,"{'id': 10194, 'name': 'Toy Story Collection', ..."
budget,60000000,0,60000000,58000000,30000000
genres,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...","[{'id': 10751, 'name': 'Family'}, {'id': 28, '...","[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...","[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...","[{'id': 10751, 'name': 'Family'}, {'id': 35, '..."
homepage,https://mgm.com/movies/goldeneye,,https://www.20thcenturystudios.com/movies/heat,,http://toystory.disney.com/toy-story
id,710,45325,949,11860,862
imdb_id,tt0113189,tt0112302,tt0113277,tt0114319,tt0114709
origin_country,[GB],[US],[US],[US],[US]
original_language,en,en,en,en,en


Hacemos un merge con el DataFrame de IDs originales (`movieId`, `tmdbId`) para que cada película tenga tanto su identificador de MovieLens como su identificador de TMDB en el dataset final.

In [12]:
final_df = pd.merge(fetched, mdf, on='id')
final_df.head().transpose()

,0,1,2,3,4
adult,False,False,False,False,False
backdrop_path,/fIWsCpYR9iGDMSbMTSAzy8L7Kg5.jpg,/jP8lHNHD89xaRPfAdyz5KEVYcSb.jpg,/xKsnZDERG1dk95wuZ5q9iks3OL3.jpg,/oMGV48EGhsNavC1PL8HMeWs5Udq.jpg,/3Rfvhy1Nl6sSGJwyjb0QiZzZYlB.jpg
belongs_to_collection,"{'id': 645, 'name': 'James Bond Collection', '...",None,"{'id': 1048282, 'name': 'Heat Collection', 'po...",None,"{'id': 10194, 'name': 'Toy Story Collection', ..."
budget,60000000,0,60000000,58000000,30000000
genres,"[{'id': 12, 'name': 'Adventure'}, {'id': 28, '...","[{'id': 10751, 'name': 'Family'}, {'id': 28, '...","[{'id': 80, 'name': 'Crime'}, {'id': 18, 'name...","[{'id': 10749, 'name': 'Romance'}, {'id': 18, ...","[{'id': 10751, 'name': 'Family'}, {'id': 35, '..."
homepage,https://mgm.com/movies/goldeneye,,https://www.20thcenturystudios.com/movies/heat,,http://toystory.disney.com/toy-story
id,710,45325,949,11860,862
imdb_id,tt0113189,tt0112302,tt0113277,tt0114319,tt0114709
origin_country,[GB],[US],[US],[US],[US]
original_language,en,en,en,en,en


In [13]:
final_df[['title', 'id']].isna().sum()

title    0
id       0
dtype: int64

In [14]:
final_df.shape 

(86242, 27)

El dataset final contiene los metadatos completos de 86,242 películas para las cuales se pudo obtener información de TMDB. Guardamos este dataset como archivo Parquet para continuar con el preprocesamiento en los siguientes notebooks.

In [15]:
final_df.to_parquet('../data/raw/movies_metadata.parquet', index=False)

## Resumen del Notebook

En este notebook se descargaron los metadatos completos de las películas del dataset MovieLens utilizando la API de TMDB con peticiones concurrentes.

**Acciones realizadas:**
- Limpieza del archivo `links.csv`: eliminación de duplicados, valores nulos e IDs inválidos
- Descarga de metadatos (título, géneros, sinopsis, fechas, puntuaciones, posters) para cada película
- Merge con los IDs originales de MovieLens para mantener la trazabilidad

**Output generado:** `ml/data/raw/movies_metadata.parquet`